In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import os
import flap

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes import data

In [ ]:
shots=np.loadtxt("/home/molnarbalazs/data/BES_ML_modelling/evaluated_e_ids.txt", dtype=str)

In [ ]:
for shot in shots[113:]:
    path='/data2/W7-X/processed_data/APDCAM/flap_recon/'
    try:
        file_list=os.listdir(os.path.join(path,shot))
    except FileNotFoundError:
        raise ValueError("Directory " + os.path.join(path,shot) + " does not exist")

    file_name_light=[i for i in file_list if ('light_orig' in i)]
    file_name_light=file_name_light[0]
    light=flap.load(os.path.join(os.path.join(path,shot),file_name_light))

    r_coord=light.coordinate('Device R')[0][0]
    time_instances_light=light.coordinate('Time')[0][:,0]
    light_data=light.data

    plt.figure(figsize=(15,3))
    if time_instances_light.shape[0]>10000:
        plt.pcolormesh(time_instances_light[::100], r_coord, light_data[::100].T, cmap='coolwarm')
    else:
        plt.pcolormesh(time_instances_light, r_coord, light_data.T, cmap='coolwarm')
    plt.savefig("/home/molnarbalazs/data/BES_ML_modelling/W7X_experimental_data_from_flap/time_evolution_shot_"+shot.replace(".", "")+".png", bbox_inches='tight')

    grid=r_coord
    energy=0
    species="Na"
    ID="we_"+shot
    verbose="W7X experimental data shot "+shot+", full shot, neural network density prediction"
    zeff=0
    q=0
    temperature=np.array(0)
    tags=['Time instance ' + str(i) + ' s' for i in time_instances_light]
    test_data=data.besInferenceDatapoints(grid=grid,energy=energy,species=species,ID=ID,zeff=zeff,q=q,temperature=temperature,verbose=verbose)
    density_data=np.zeros((len(time_instances_light),len(r_coord)))
    test_data.add_datapoints_bulk(density_data, light_data, tags)
    test_data.export_to_h5(path_to_dir="/home/molnarbalazs/data/BES_ML_modelling/W7X_experimental_data_from_flap")